# Lab Ngày 08: Học chủ động cho bộ phát hiện xe trên video ban đêm

Mỗi lần chạy notebook tương ứng với một vòng học chủ động (active learning):

```
day8_data.zip ── tải lên ──> đánh giá khởi đầu lạnh
                             tinh chỉnh trên toàn bộ nhãn đã sửa
                             đánh giá trên tập kiểm thử
                             chọn lô ảnh mới theo độ bất định
day8_roundN_out.zip <── tải về ──  đóng gói lô ảnh kèm nhãn gợi ý

Trên máy của bạn: sửa nhãn lô mới (AnyLabeling, CVAT, SAM hoặc sửa tay),
chạy python3 tools/pack_labels.py to_label/roundN để tạo day8_data.zip mới, rồi chạy lại notebook.
```

| Lần chạy | Nhãn có trong file zip | Việc notebook làm |
| --- | --- | --- |
| 1 | chưa có | đánh giá `yolov8n` khởi đầu lạnh (cold start), chọn lô vòng 1 |
| 2 | `labels/round1/` | tinh chỉnh (fine-tune) trên vòng 1, so với khởi đầu lạnh, chọn lô vòng 2 |
| 3 | `labels/round1/`, `labels/round2/` | tinh chỉnh trên vòng 1 và 2, so sánh, chọn lô vòng 3 |
| ... | ... | dừng khi kết quả không còn cải thiện đáng kể |

Bạn không cần khai báo đang ở vòng mấy: notebook tự đếm các thư mục `labels/round*/` trong file
zip được tải lên.

Trước khi chạy, vào **Runtime > Change runtime type** và chọn GPU nếu Colab cấp, sau đó chọn
**Runtime > Run all**.

In [ ]:
STUDENT_NAME = "Họ và tên"
DATA_ZIP = ""          # để trống: tải file lên ở ô 1; hoặc điền đường dẫn file zip đã có trên Colab/Drive

AL_K = 12              # số ảnh chọn mỗi vòng (8-16)
MIN_GAP_S = 2.0        # khoảng cách thời gian tối thiểu (giây) giữa hai ảnh trong cùng một lô
STRATEGY = "uncertainty"   # "uncertainty" (theo độ bất định) hoặc "random" (ngẫu nhiên, để đối chứng)
W_U, W_A, W_D = 0.5, 0.3, 0.2   # trọng số: độ bất định / số box mơ hồ / độ đa dạng theo thời gian

BASE_MODEL = "yolov8n.pt"      # mô hình khởi đầu lạnh, cũng là điểm xuất phát khi tinh chỉnh
COCO_VEHICLE = [2, 5, 7]       # lớp car, bus, truck của COCO, gộp thành lớp `car` của lab
IMGSZ = 960
EPOCHS = 50
PRELABEL_CONF = 0.25           # ngưỡng độ tin cậy của box được đưa vào nhãn gợi ý
SEED = 8


## 0. Cài đặt thư viện

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.161"])
print("Đã cài xong ultralytics.")


## 1. Tải lên `day8_data.zip`

Trên máy của bạn, tạo file zip bằng lệnh `python3 tools/make_data_zip.py` ở lần đầu. Từ vòng thứ
hai, `pack_labels.py` sẽ tự tạo lại file này sau mỗi lần bạn sửa nhãn. Lần chạy nào cũng cần tải
lên bản mới nhất, vì notebook không lưu lại gì giữa các lần chạy.

Nếu tải lên qua nút chọn file quá chậm, bạn có thể kéo file zip vào bảng **Files** (biểu tượng thư
mục ở thanh bên trái) hoặc để trong Google Drive, rồi đặt `DATA_ZIP = "/content/day8_data.zip"`
(hoặc đường dẫn trên Drive) ở ô cấu hình.

In [ ]:
import shutil
import zipfile
from pathlib import Path

WORK = Path("/content")
LAB = WORK / "day8"

if DATA_ZIP:
    zip_path = Path(DATA_ZIP)
    if not zip_path.exists():
        raise SystemExit(f"Không tìm thấy {zip_path}. Để trống DATA_ZIP nếu muốn tải file lên bằng nút chọn file.")
else:
    try:
        from google.colab import files
        print("Chọn file day8_data.zip:")
        uploaded = files.upload()
        zip_path = WORK / next(iter(uploaded))
    except ImportError:
        zip_path = Path("day8_data.zip").resolve()
print(f"Dùng {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")

if LAB.exists():
    shutil.rmtree(LAB)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORK)
if not (LAB / "data" / "pool" / "images").is_dir():
    raise SystemExit(f"{zip_path.name} không chứa day8/data/pool/images. Hãy tạo lại bằng tools/make_data_zip.py.")

sys.path.insert(0, str(LAB / "tools"))
print("Thư mục lab:", LAB)


## 2. Dữ liệu và các vòng đã gán nhãn

In [ ]:
import csv
import json
import re

import al_select
from yolo_io import read_yolo

POOL = LAB / "data" / "pool" / "images"
TEST = LAB / "data" / "test" / "images"
OUT = LAB / "outputs"
OUT.mkdir(exist_ok=True)

times = {r["file"]: float(r["t_sec"]) for r in csv.DictReader(open(LAB / "data" / "frames.csv"))}
pool_files = sorted(p.name for p in POOL.glob("*.jpg"))
test_paths = sorted(TEST.glob("*.jpg"))
ref = {p.name: read_yolo(LAB / "data" / "test" / "labels" / f"{p.stem}.txt") for p in test_paths}

found = sorted(int(m.group(1)) for d in (LAB / "labels").glob("round*")
               if (m := re.fullmatch(r"round(\d+)", d.name)) and (d / "batch.json").exists())
if found != list(range(1, len(found) + 1)):
    raise SystemExit(f"Thư mục labels/ có các vòng {found}, nhưng các vòng phải liên tục từ vòng 1.")
R = len(found)

labeled, label_path, strategies = {}, {}, []
for r in found:
    batch = json.loads((LAB / "labels" / f"round{r}" / "batch.json").read_text())
    strategies.append(batch["strategy"])
    for name in batch["files"]:
        if name not in pool_files:
            raise SystemExit(f"Ảnh {name} (vòng {r}) không thuộc tập pool.")
        label_path[name] = LAB / "labels" / f"round{r}" / f"{Path(name).stem}.txt"
        labeled[name] = read_yolo(label_path[name])

al_select.W_U, al_select.W_A, al_select.W_D = W_U, W_A, W_D
print(f"Pool: {len(pool_files)} ảnh. Tập kiểm thử: {len(test_paths)} ảnh, {sum(map(len, ref.values()))} box tham chiếu.")
print(f"Các vòng đã gán nhãn: {found or 'chưa có'}, tổng {len(labeled)} ảnh, {sum(map(len, labeled.values()))} box.")
print("Lần chạy này:", "chỉ đánh giá khởi đầu lạnh" if R == 0 else f"tinh chỉnh trên vòng 1 đến {R}", f"rồi chọn lô vòng {R + 1}.")


## 3. Kiểm tra GPU và định nghĩa hàm dự đoán

In [ ]:
import datetime as dt

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("Thiết bị:", GPU)
if DEVICE == "cpu":
    print("Không có GPU: notebook vẫn chạy được nhưng tinh chỉnh sẽ chậm. Thử Runtime > Change runtime type > GPU; nếu chưa có, báo Lab Coach.")


def predict(model, paths, coco=False, conf=0.01):
    kw = dict(imgsz=IMGSZ, conf=conf, iou=0.6, max_det=300, verbose=False, device=DEVICE)
    if coco:
        kw.update(classes=COCO_VEHICLE, agnostic_nms=True)
    out = {}
    for i in range(0, len(paths), 16):
        chunk = paths[i:i + 16]
        for p, r in zip(chunk, model.predict([str(x) for x in chunk], **kw)):
            h, w = r.orig_shape
            boxes = []
            for b in r.boxes:
                x1, y1, x2, y2 = (float(v) for v in b.xyxy[0])
                boxes.append({"cls": 0, "cx": (x1 + x2) / 2 / w, "cy": (y1 + y2) / 2 / h,
                              "w": (x2 - x1) / w, "h": (y2 - y1) / h, "conf": round(float(b.conf[0]), 4)})
            out[p.name] = boxes
    return out


## 4. Mô hình khởi đầu lạnh (cold start)

Đây là `yolov8n` đã huấn luyện sẵn (pretrained) trên COCO, chưa được huấn luyện thêm trên video
này. Ba lớp `car`, `bus`, `truck` của COCO đều được tính là `car`. Kết quả ở bước này là mốc để so
sánh với mọi vòng sau.

Mọi vòng đều được chấm trên cùng 20 ảnh kiểm thử với cùng bộ nhãn tham chiếu. Box tham chiếu cao
dưới 16 pixel (xe ở sát đường chân trời) được bỏ qua khi chấm, xem `data/DATA.md`.

In [ ]:
from det_eval import evaluate


def save_metrics(round_no, desc, n_img, n_box, test_metrics):
    payload = {
        "round": round_no,
        "model": desc,
        "base_model": BASE_MODEL,
        "n_train_images": n_img,
        "n_train_boxes": n_box,
        "epochs": 0 if round_no == 0 else EPOCHS,
        "imgsz": IMGSZ,
        "strategies": strategies[:round_no],
        "test": test_metrics,
        "reference": "data/test/labels",
        "runtime": GPU,
        "student": STUDENT_NAME,
        "created": dt.datetime.now().isoformat(timespec="seconds"),
    }
    path = OUT / f"metrics_round{round_no}.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    return payload


cold = YOLO(BASE_MODEL)
cold_desc = "yolov8n cold start (COCO car+bus+truck)"
pred_test_cold = predict(cold, test_paths, coco=True)
m0 = save_metrics(0, cold_desc, 0, 0, evaluate(ref, pred_test_cold))
t = m0["test"]
print(f"Khởi đầu lạnh: AP50 {t['ap50']:.3f}  P {t['precision']:.3f}  R {t['recall']:.3f}  F1 {t['f1']:.3f}")
print("Độ phủ theo kích thước xe:", {k: v["recall"] for k, v in t["recall_by_size"].items()})


## 5. Tinh chỉnh trên toàn bộ nhãn đã sửa (bỏ qua ở lần chạy đầu)

Ở mỗi vòng, mô hình được huấn luyện lại từ `yolov8n.pt` trên toàn bộ nhãn từ vòng 1 đến vòng hiện
tại, thay vì huấn luyện tiếp từ mô hình của vòng trước. Cách này giúp so sánh các vòng một cách
công bằng. Vì số nhãn còn ít, lab không tách riêng tập kiểm định (validation set): mô hình được
huấn luyện với `val=False`, dùng trọng số `last.pt`, và chỉ được đánh giá trên 20 ảnh kiểm thử.

Khi huấn luyện xong, Ultralytics tự in ra một bảng có cột `mAP50`. Bảng này được tính trên chính
các ảnh huấn luyện nên luôn cao, không dùng cho báo cáo. Kết quả cần dùng là dòng `Vòng N: AP50 ...`
in ngay bên dưới.

In [ ]:
if R == 0:
    model = cold
    model_desc = cold_desc
    pred_test = pred_test_cold
    is_coco = True
    print("Chưa có nhãn nào, dùng mô hình khởi đầu lạnh để chọn lô vòng 1.")
else:
    ds = WORK / "work" / f"train_round{R}"
    if ds.exists():
        shutil.rmtree(ds)
    (ds / "images").mkdir(parents=True)
    (ds / "labels").mkdir(parents=True)
    for name in labeled:
        shutil.copy2(POOL / name, ds / "images" / name)
        shutil.copy2(label_path[name], ds / "labels" / f"{Path(name).stem}.txt")
    (ds / "data.yaml").write_text(f"path: {ds}\ntrain: images\nval: images\nnames:\n  0: car\n")

    trainer = YOLO(BASE_MODEL)
    trainer.train(data=str(ds / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=16, seed=SEED,
                  val=False, plots=False, project=str(WORK / "runs"), name=f"round{R}",
                  exist_ok=True, device=DEVICE, verbose=False)
    model = YOLO(str(WORK / "runs" / f"round{R}" / "weights" / "last.pt"))
    model_desc = f"yolov8n fine-tune vong 1..{R}"
    is_coco = False
    pred_test = predict(model, test_paths)
    mR = save_metrics(R, model_desc, len(labeled), sum(map(len, labeled.values())), evaluate(ref, pred_test))
    t = mR["test"]
    print(f"Vòng {R}: AP50 {t['ap50']:.3f}  P {t['precision']:.3f}  R {t['recall']:.3f}  F1 {t['f1']:.3f}")


## 6. So sánh khởi đầu lạnh với học chủ động

Bảng dưới đọc toàn bộ các file `outputs/metrics_round*.json`, trong đó số liệu của các vòng trước
nằm sẵn trong file zip bạn tải lên. Ảnh so sánh gồm ba cột: nhãn tham chiếu, mô hình khởi đầu lạnh
và mô hình của vòng này. Trên các cột mô hình, box xanh lá là phát hiện đúng, box đỏ là phát hiện
nhầm (false positive) và box vàng là xe bị bỏ sót (false negative).

In [ ]:
from IPython.display import Image as IPImage, display

import viz

history = sorted((json.loads(f.read_text()) for f in OUT.glob("metrics_round*.json")), key=lambda m: m["round"])
base = history[0]["test"]["ap50"]
print(f"{'Vòng':>4} {'Ảnh train':>9} {'AP50':>6} {'ΔAP50':>7} {'P':>6} {'R':>6} {'R nhỏ':>8}")
for m in history:
    t = m["test"]
    d = "" if m["round"] == 0 else f"{t['ap50'] - base:+.3f}"
    print(f"{m['round']:>4} {m['n_train_images']:>9} {t['ap50']:>6.3f} {d:>7} {t['precision']:>6.3f} "
          f"{t['recall']:>6.3f} {str(t['recall_by_size']['small']['recall']):>8}")

show = [p for p in test_paths if p.stem in ("frame_0050", "frame_0150", "frame_0250", "frame_0350")] or test_paths[:4]
cols = [("cold start", pred_test_cold)] + ([(f"round {R}", pred_test)] if R else [])
grid = OUT / f"compare_round{R}.jpg"
viz.compare_grid(show, ref, cols, grid)
display(IPImage(str(grid), width=1100))

if R >= 1:
    prev = next((m for m in history if m["round"] == R - 1), None)
    gain_prev = mR["test"]["ap50"] - prev["test"]["ap50"] if prev else None
    print(f"So với khởi đầu lạnh: {mR['test']['ap50'] - base:+.3f} AP50" +
          ("" if gain_prev is None else f". So với vòng {R - 1}: {gain_prev:+.3f} AP50."))
    if gain_prev is not None and gain_prev < 0.01:
        print("Mức tăng dưới 0.01 AP50 so với vòng trước. Hãy xem ảnh so sánh và cân nhắc dừng hoặc đổi chiến lược chọn mẫu.")


## 7. Lấy mẫu theo độ bất định (uncertainty sampling) cho vòng tiếp theo

Mô hình hiện tại (ở lần chạy đầu là mô hình khởi đầu lạnh) dự đoán trên mọi ảnh pool chưa được gán
nhãn. Độ bất định của mỗi box được tính bằng `u = 1 - |2·conf - 1|`, đạt cao nhất khi độ tin cậy
(confidence) bằng 0.5. Điểm của một ảnh là:

`score = W_U · U + W_A · A + W_D · D`

trong đó:

- `U` là trung bình độ bất định của 5 box khó nhất trong ảnh;
- `A` là số box mơ hồ (0.15 ≤ conf < 0.5), chuẩn hoá theo giá trị lớn nhất trong pool;
- `D` là khoảng cách thời gian từ ảnh đó tới ảnh đã gán nhãn gần nhất, tính tối đa 10 giây, thể
  hiện độ đa dạng (diversity).

Các ảnh được chọn theo điểm từ cao xuống thấp, với điều kiện hai ảnh trong cùng một lô cách nhau ít
nhất `MIN_GAP_S` giây. Lý do là camera đứng yên, nên hai ảnh quá gần nhau gần như trùng lặp. Chi
tiết cài đặt nằm trong `tools/al_select.py`.

In [ ]:
import csv as _csv

from al_batch import write_batch

candidates = [POOL / n for n in pool_files if n not in labeled]
pred_pool = predict(model, candidates, coco=is_coco)
scored = al_select.score_frames(pred_pool, times, [times[n] for n in labeled])
selected = al_select.select_batch(scored, k=min(AL_K, len(scored)), min_gap_s=MIN_GAP_S, strategy=STRATEGY, seed=SEED + R)

NEXT = R + 1
sel_csv = OUT / f"selection_round{NEXT}.csv"
with sel_csv.open("w", newline="") as fh:
    w = _csv.DictWriter(fh, fieldnames=["rank", "file", "t_sec", "score", "U", "A", "D", "n_boxes", "n_ambiguous", "empty", "selected"])
    w.writeheader()
    for row in scored:
        w.writerow({k: row[k] for k in w.fieldnames})
viz.selection_sheet(POOL, selected, OUT / f"selection_round{NEXT}.jpg")

batch_dir = write_batch(NEXT, selected, scored, POOL, pred_pool, LAB / "to_label" / f"round{NEXT}",
                        model_desc, STRATEGY, AL_K, MIN_GAP_S, PRELABEL_CONF)
n_pre = sum(1 for n in (r["file"] for r in selected) for b in pred_pool[n] if b["conf"] >= PRELABEL_CONF)
print(f"Lô vòng {NEXT}: {len(selected)} ảnh (chiến lược {STRATEGY}), {n_pre} box gợi ý, "
      f"trải từ giây {selected[0]['t_sec']:.1f} đến giây {selected[-1]['t_sec']:.1f}.")
display(IPImage(str(OUT / f"selection_round{NEXT}.jpg"), width=1100))


## 8. Tải về kết quả và lô ảnh cần gán nhãn

Giải nén `day8_roundN_out.zip` vào thư mục gốc repo của bạn, các file sẽ tự nằm đúng chỗ
(`outputs/` và `to_label/roundN+1/`). Sau đó sửa nhãn trong `to_label/roundN+1/` (hướng dẫn có trong
file `HUONG_DAN.txt` của thư mục đó) và chạy `python3 tools/pack_labels.py to_label/roundN+1`.

In [ ]:
import summarize_rounds
summarize_rounds.ROOT = LAB
summarize_rounds.main()
rounds_table = LAB / "reports" / "rounds_table.md"

out_zip = WORK / f"day8_round{R}_out.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    keep = [OUT / "metrics_round0.json", OUT / f"metrics_round{R}.json", grid, sel_csv, OUT / f"selection_round{NEXT}.jpg"]
    for f in dict.fromkeys(keep):
        if f.exists():
            zf.write(f, f"outputs/{f.name}")
    zf.write(rounds_table, "reports/rounds_table.md")
    for f in sorted(batch_dir.rglob("*")):
        if f.is_file():
            zf.write(f, f.relative_to(LAB).as_posix())
print(f"Đã tạo {out_zip.name} ({out_zip.stat().st_size / 2**20:.1f} MB).")
try:
    from google.colab import files
    files.download(str(out_zip))
except ImportError:
    pass
